# Session 11 — Map validation and area estimation
*Séance 11 — Validation des cartes et estimation des superficies*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/ML4EO_Bootcamp/blob/main/notebooks/11_validation_cartes.ipynb)

**Course:** Machine Learning for Earth Observation (45 h) · **Instructor:** Noé Carème Fouotsa Manfouo

**By the end of this session you will be able to:**

1. Compute overall, user's and producer's accuracies and F1-scores from a confusion matrix.
2. Explain why pixel counting on a map gives biased areas.
3. Estimate class areas and 95% confidence intervals with a stratified estimator (Olofsson et al., 2014).
4. Plan a validation sample.

**How to use this notebook:** read the lecture cells, run every code cell (`Shift + Enter`), then complete the cells marked
`# YOUR CODE HERE` in the lab section. Run the ✅ check cells to get instant feedback.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
import sys, pathlib, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("ML4EO_Bootcamp").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/ML4EO_Bootcamp.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *"geopandas rasterio rioxarray".split()], check=True)
    ROOT = pathlib.Path("ML4EO_Bootcamp")
else:
    ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data"
sys.path.append(str(ROOT / "notebooks"))   # course helper module (ml4eo_data.py)


def check(condition, message="Check failed"):
    """Instant feedback: raises an error with a hint if the condition is False."""
    assert condition, message
    print("✅ Passed")


def check_equal(value, expected, message=None):
    assert value == expected, message or f"Expected {expected!r}, got {value!r}"
    print("✅ Passed")

## Part A — Lecture

A map is a **model**: every class contains errors. Good practice (Olofsson et al., 2014) separates three steps:
1. **Sampling design** — usually stratified random sampling, using the map classes as strata;
2. **Response design** — how the reference label of each sample is decided (field visit, high-resolution imagery);
3. **Analysis** — accuracies *and* area estimates with confidence intervals.

Below: a forest-change map (30 m Landsat pixels, 0.09 ha) and a stratified sample of 640 reference points. The numbers
are inspired by the worked example of Olofsson et al. (2014). Rows = **map** class, columns = **reference** class.

In [ ]:
import numpy as np, pandas as pd

classes = ["Deforestation", "Forest gain", "Stable forest", "Stable non-forest"]
mapped_pixels = np.array([200_000, 150_000, 3_200_000, 6_450_000])
counts = np.array([[66,  0,   5,   4],
                   [ 0, 55,   8,  12],
                   [ 1,  0, 153,  11],
                   [ 2,  1,   9, 313]])
cm = pd.DataFrame(counts, index=[f"map: {c}" for c in classes], columns=[f"ref: {c}" for c in classes])
cm

In [ ]:
overall = np.trace(counts) / counts.sum()
users = np.diag(counts) / counts.sum(axis=1)       # 1 - commission error
producers = np.diag(counts) / counts.sum(axis=0)   # 1 - omission error (sample-based, unweighted!)
pd.DataFrame({"user's accuracy": users, "producer's accuracy (unweighted)": producers}, index=classes).round(3)

### Why sample proportions are not enough
The sample over-represents rare classes on purpose. Each map class *h* must be **weighted** by its share of the map,
$W_h = A_{m,h} / A_{tot}$. The estimated proportion of each cell of the population matrix is
$\hat p_{ij} = W_i \, n_{ij} / n_{i\cdot}$, the estimated area of class *j* is $\hat A_j = A_{tot} \sum_i \hat p_{ij}$, and its standard error is
$S(\hat p_{\cdot j}) = \sqrt{\sum_i W_i^2 \, \frac{(n_{ij}/n_{i\cdot})(1 - n_{ij}/n_{i\cdot})}{n_{i\cdot} - 1}}$.

## Part B — Lab

**Task 1 — Weights and population matrix.** Compute `W` (map-class weights, sum = 1) and `p` = the estimated population
error matrix (same shape as `counts`, sums to 1).

In [ ]:
# YOUR CODE HERE
W = None
p = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(abs(W.sum() - 1) < 1e-9)
check(abs(p.sum() - 1) < 1e-9)

**Task 2 — Area estimates.** Compute the estimated area of each class in **hectares** (`area_ha`, one 30 m pixel = 0.09 ha),
its standard error (`se_ha`) and the 95% confidence interval half-width (`ci95_ha` = 1.96 × SE).

In [ ]:
# YOUR CODE HERE
area_ha = None
se_ha = None
ci95_ha = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(abs(area_ha.sum() - mapped_pixels.sum() * 0.09) < 1, "Estimated areas must add up to the total area")
check(area_ha[0] > mapped_pixels[0] * 0.09, "Deforestation is under-mapped here: omission errors in the stable classes")

**Task 3 — Weighted accuracies.** Compute the **weighted** overall accuracy `oa_w` and producer's accuracies `pa_w`
from `p`, and compare `pa_w` with the unweighted `producers`.

In [ ]:
# YOUR CODE HERE
oa_w = None
pa_w = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(0.8 < oa_w < 1)
check(pa_w[0] < producers[0], "Weighting reveals the true omission of deforestation")

**Task 4 — F1-score.** Write `f1(ua, pa)` (harmonic mean) and compute `f1_scores` per class using `users` and `pa_w`.

In [ ]:
# YOUR CODE HERE
def f1(ua, pa):
    pass

f1_scores = None
raise NotImplementedError()

In [ ]:
# ✅ Check (do not modify)
check(f1(1, 1) == 1 and abs(f1(0.5, 1) - 2/3) < 1e-9)

### Planning a sample
A common rule (Cochran) for stratified sampling: $n = \left(\sum_i W_i S_i / S(\hat O)\right)^2$ with $S_i = \sqrt{U_i(1-U_i)}$,
where $U_i$ are the expected user's accuracies and $S(\hat O)$ the target standard error of overall accuracy (e.g. 0.01).
Then allocate at least 50–100 points to each rare class.

In [ ]:
expected_ua = np.array([0.7, 0.6, 0.9, 0.95])
S = np.sqrt(expected_ua * (1 - expected_ua))
n_total = (np.sum(W * S) / 0.01) ** 2
print(f"Total sample size for SE(OA) = 0.01: {n_total:.0f} points")

**❓ Question —** Why is kappa no longer recommended for reporting map accuracy (Pontius & Millones, 2011)? What would you report instead?

*Your answer here.*

## Summary

- Report user's and producer's accuracies per class, not only overall accuracy.
- Pixel counting is biased; use a stratified estimator to get areas with confidence intervals.
- Design the validation sample before looking at the results.

**To go further:**

- Olofsson, P. et al. (2014). Good practices for estimating area and assessing accuracy of land change. *Remote Sensing of Environment*, 148, 42–57.
- Pontius, R. G. & Millones, M. (2011). Death to Kappa. *International Journal of Remote Sensing*, 32(15), 4407–4429.